In [0]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from databricks.sdk import WorkspaceClient

# ==========================================
# CONFIGURATION
# ==========================================
# 1. Set this to True the FIRST time you run it. 
# 2. After it works, change it to False and delete your private details below.
SETUP_MODE = False  

# ⚠️ ONLY USED WHEN SETUP_MODE = True
MY_GMAIL_ADDRESS = "hidden"
MY_APP_PASSWORD = "hidden" 
RECIPIENT_ADDRESS = "hidden"


# ==========================================
# STEP 1: CONDITIONAL VAULT STORAGE
# ==========================================
if SETUP_MODE:
    w = WorkspaceClient()
    scope_name = "email_creds"

    try:
        w.secrets.create_scope(scope=scope_name)
    except:
        pass # Scope already exists

    # Save the real credentials to the vault
    w.secrets.put_secret(scope=scope_name, key="gmail_username", string_value=MY_GMAIL_ADDRESS)
    w.secrets.put_secret(scope=scope_name, key="gmail_password", string_value=MY_APP_PASSWORD)
    w.secrets.put_secret(scope=scope_name, key="gmail_receiver", string_value=RECIPIENT_ADDRESS)
    print("🔒 Setup Mode Active: Credentials successfully saved/updated in Secrets Vault!")
else:
    print("🤖 Production Mode Active: Skipping setup, reading directly from secure vault.")

# Wipe variables from local memory just to be safe
MY_GMAIL_ADDRESS = MY_APP_PASSWORD = RECIPIENT_ADDRESS = None


# ==========================================
# STEP 2: FETCH SECRETS & SEND EMAIL
# ==========================================
def send_good_morning():
    # This safely pulls from the vault every single time, no matter what
    sender = dbutils.secrets.get(scope="email_creds", key="gmail_username")
    password = dbutils.secrets.get(scope="email_creds", key="gmail_password")
    receiver = dbutils.secrets.get(scope="email_creds", key="gmail_receiver")

    msg = MIMEMultipart()
    msg['From'] = sender
    msg['To'] = receiver
    msg['Subject'] = "Good Morning! ☀️"
    msg.attach(MIMEText("Good morning! Have a wonderful day ahead.", 'plain'))

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender, password)
        server.sendmail(sender, receiver, msg.as_string())
        print("🚀 Success! Email sent beautifully.")
    except Exception as e:
        print(f"Error sending email: {e}")
    finally:
        try: server.quit()
        except: pass

# Run the function
send_good_morning()